###1) Konfiguracja

In [0]:
from pyspark.sql import functions as F

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

BACKTEST_TABLE = (
    "workspace.energy_gold."
    "battery_dispatch_backtest_15m"
)

SUMMARY_TABLE = (
    "workspace.energy_gold."
    "battery_backtest_summary"
)

backtest_df = spark.table(BACKTEST_TABLE)
summary_df = spark.table(SUMMARY_TABLE)

print("Backtest:", backtest_df.count())
print("Podsumowanie:", summary_df.count())

display(
    summary_df.orderBy(
        F.desc("adjusted_net_revenue_usd")
    )
)

###2) Schema backtestu

In [0]:
backtest_df.printSchema()

###3) Dzienne wyniki dla każdego scenariusza

In [0]:
daily_raw_df = (
    backtest_df
    .withColumn(
        "delivery_date_local",
        F.to_date(
            F.from_utc_timestamp(
                "target_interval_start_utc",
                "America/Los_Angeles"
            )
        )
    )
    .groupBy(
        "delivery_date_local",
        "scenario"
    )
    .agg(
        F.count("*").alias("interval_count"),
        F.sum("net_revenue_usd").alias(
            "net_revenue_usd"
        ),
        F.sum("dam_revenue_usd").alias(
            "dam_revenue_usd"
        ),
        F.sum("realtime_adjustment_usd").alias(
            "realtime_adjustment_usd"
        ),
        F.sum("degradation_cost_usd").alias(
            "degradation_cost_usd"
        ),
        F.sum("actual_pv_energy_mwh").alias(
            "actual_pv_energy_mwh"
        ),
        F.sum("pv_to_grid_mwh").alias(
            "pv_to_grid_mwh"
        ),
        F.sum("pv_to_battery_mwh").alias(
            "pv_to_battery_mwh"
        ),
        F.sum("grid_to_battery_mwh").alias(
            "grid_to_battery_mwh"
        ),
        F.sum("battery_to_grid_mwh").alias(
            "battery_to_grid_mwh"
        ),
        F.sum(
            F.greatest(
                F.col("net_export_mwh"),
                F.lit(0.0)
            )
        ).alias("export_mwh"),
        F.sum(
            F.greatest(
                -F.col("net_export_mwh"),
                F.lit(0.0)
            )
        ).alias("grid_import_mwh"),
        F.sum("battery_charge_mwh").alias(
            "battery_charge_mwh"
        ),
        F.sum("battery_discharge_mwh").alias(
            "battery_discharge_mwh"
        ),
        F.sum("curtailed_energy_mwh").alias(
            "curtailed_energy_mwh"
        ),
        F.sum(
            F.abs(
                F.col("net_export_mwh")
                - F.col("dam_schedule_mwh")
            )
        ).alias(
            "absolute_schedule_deviation_mwh"
        ),
        F.max_by(
            "soc_after_mwh",
            "target_interval_start_utc"
        ).alias("end_of_day_soc_mwh")
    )
)

daily_baseline_df = (
    daily_raw_df
    .filter(
        F.col("scenario") == "NO_BATTERY"
    )
    .select(
        "delivery_date_local",
        F.col("net_revenue_usd").alias(
            "no_battery_revenue_usd"
        )
    )
)

daily_results_df = (
    daily_raw_df
    .join(
        daily_baseline_df,
        on="delivery_date_local",
        how="left"
    )
    .withColumn(
        "revenue_difference_vs_no_battery_usd",
        F.col("net_revenue_usd")
        - F.col("no_battery_revenue_usd")
    )
)

display(
    daily_results_df
    .orderBy(
        "delivery_date_local",
        "scenario"
    )
)

###4) Zapis dziennych KPI do Gold

In [0]:
DAILY_RESULTS_TABLE = (
    "workspace.energy_gold."
    "battery_backtest_daily"
)

daily_output_df = (
    daily_results_df
    .withColumn(
        "_dashboard_version",
        F.lit("explicit_flows_v2")
    )
    .withColumn(
        "_created_at_utc",
        F.current_timestamp()
    )
)

(
    daily_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DAILY_RESULTS_TABLE)
)

saved_daily_df = spark.table(
    DAILY_RESULTS_TABLE
)

print(f"Zapisano: {DAILY_RESULTS_TABLE}")

display(
    saved_daily_df.agg(
        F.count("*").alias("row_count"),
        F.countDistinct(
            "delivery_date_local"
        ).alias("date_count"),
        F.countDistinct(
            "scenario"
        ).alias("scenario_count")
    )
)

###5) Końcowe testy jakości KPI

In [0]:
saved_daily_df = spark.table(
    "workspace.energy_gold."
    "battery_backtest_daily"
)

saved_summary_df = spark.table(
    "workspace.energy_gold."
    "battery_backtest_summary"
)

duplicate_count = (
    saved_daily_df
    .groupBy(
        "delivery_date_local",
        "scenario"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

expected_daily_counts_df = (
    backtest_df
    .filter(F.col("scenario") == "NO_BATTERY")
    .withColumn(
        "delivery_date_local",
        F.to_date(
            F.from_utc_timestamp(
                "target_interval_start_utc",
                "America/Los_Angeles"
            )
        )
    )
    .groupBy("delivery_date_local")
    .agg(
        F.count("*").alias("expected_interval_count")
    )
)

invalid_interval_count = (
    saved_daily_df
    .join(
        expected_daily_counts_df,
        on="delivery_date_local",
        how="left"
    )
    .filter(
        F.col("expected_interval_count").isNull()
        |
        (
            F.col("interval_count")
            != F.col("expected_interval_count")
        )
    )
    .count()
)

missing_value_count = (
    saved_daily_df
    .filter(
        F.col("net_revenue_usd").isNull()
        |
        F.col(
            "revenue_difference_vs_no_battery_usd"
        ).isNull()
    )
    .count()
)

invalid_baseline_difference_count = (
    saved_daily_df
    .filter(
        (F.col("scenario") == "NO_BATTERY")
        &
        (
            F.abs(
                F.col(
                    "revenue_difference_vs_no_battery_usd"
                )
            ) > 1e-9
        )
    )
    .count()
)

daily_totals_df = (
    saved_daily_df
    .groupBy("scenario")
    .agg(
        F.sum("net_revenue_usd").alias(
            "daily_total_revenue_usd"
        )
    )
)

reconciliation_error_count = (
    daily_totals_df
    .join(
        saved_summary_df.select(
            "scenario",
            "raw_net_revenue_usd"
        ),
        on="scenario",
        how="inner"
    )
    .filter(
        F.abs(
            F.col("daily_total_revenue_usd")
            - F.col("raw_net_revenue_usd")
        ) > 1e-6
    )
    .count()
)

test_results = {
    "duplicate_count": duplicate_count,
    "invalid_interval_count":
        invalid_interval_count,
    "missing_value_count":
        missing_value_count,
    "invalid_baseline_difference_count":
        invalid_baseline_difference_count,
    "reconciliation_error_count":
        reconciliation_error_count,
}

for name, value in test_results.items():
    print(f"{name}: {value}")

if any(test_results.values()):
    raise RuntimeError(
        "Testy jakości KPI nie zostały zaliczone."
    )

print("Wszystkie testy KPI zostały zaliczone.")